[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C37_MLOps_Course/03_cicd_gates/03_cicd_gates.ipynb)

# 03 · 从零实现评测门禁与显著性检验

本 notebook 从零造一个评测门禁：**绝对阈值 + 回归检测 + bootstrap 置信区间 + 置换检验 + 规则聚合**，用 **numpy + 标准库** 实现，过 `assert`。核心是把「这点提升是真的吗」用统计严谨地回答。

**路线**：① 评测与采样噪声 → ② 绝对阈值规则 → ③ 回归检测 → ④ bootstrap 置信区间 → ⑤ 置换检验 + 规则聚合 → ✏️ 4 道练习 → 📖 答案 → 🧪 真实评测：两个分类器的显著性对比。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
print('环境就绪 ✅ | numpy', np.__version__)

## 1 · 先建立直觉：指标本身带采样噪声

测试集是从真实分布抽样的有限样本，**在它上面算的任何指标都是随机变量**。

用同一个「真实准确率 0.90」的模型，反复抽不同的 1000 条测试样本评测，看 acc 会抖多大——这就是 0.5% 的差异可能纯属噪声的原因。

In [ ]:
def eval_acc(true_acc, n, rng):
    # 模拟：每条样本以 true_acc 概率被预测对，返回这批 n 条的经验准确率
    correct = rng.random(n) < true_acc
    return correct.mean()

# 同一个 true_acc=0.90 的模型，评 30 次（每次新抽 1000 条）
accs = [eval_acc(0.90, 1000, rng) for _ in range(30)]
print(f'真实 acc=0.900，但 30 次评测的经验 acc 范围 = [{min(accs):.3f}, {max(accs):.3f}]')
print(f'标准差 ≈ {np.std(accs):.4f}  (理论 sqrt(p(1-p)/n) = {np.sqrt(0.9*0.1/1000):.4f})')
assert max(accs) - min(accs) > 0.01, '同一个模型，不同测试集就能差 1%+ —— 这就是噪声'
print('✅ 关键直觉：1% 量级的指标差异，完全可能只是采样噪声')

## 2 · 绝对阈值规则：守底线（带方向）

最基本的门禁规则：指标必须达到硬阈值。方向是参数——acc/recall 越大越好(`max`)，latency/error 越小越好(`min`)。

In [ ]:
def rule_abs_threshold(value, tau, mode='max'):
    '''返回 (passed, reason)。'''
    if mode == 'max':
        ok = value >= tau
        reason = f'{value:.4f} {">=" if ok else "<"} 阈值 {tau} (mode=max)'
    else:
        ok = value <= tau
        reason = f'{value:.4f} {"<=" if ok else ">"} 阈值 {tau} (mode=min)'
    return ok, reason

ok1, r1 = rule_abs_threshold(0.918, 0.85, 'max')
ok2, r2 = rule_abs_threshold(0.80, 0.85, 'max')
ok3, r3 = rule_abs_threshold(85.0, 100.0, 'min')      # 延迟 85ms <= 100ms
print('acc 0.918 vs 0.85:', ok1, '|', r1)
print('acc 0.80  vs 0.85:', ok2, '|', r2)
print('lat 85ms vs 100ms:', ok3, '|', r3)
assert ok1 and not ok2 and ok3
print('✅ 绝对阈值规则正确（双向：max 守下限、min 守上限）')

## 3 · 回归检测：别让模型偷偷退步

候选哪怕过了绝对阈值，若比现役基线退步太多也要拦。`eps` 是容忍的回归幅度（允许用一点 acc 换更小/更快的模型）。

In [ ]:
def rule_no_regression(candidate, baseline, eps=0.0, mode='max'):
    delta = candidate - baseline if mode == 'max' else baseline - candidate
    ok = delta >= -eps                              # 退步不超过 eps
    reason = f'Δ(候选-基线)={candidate-baseline:+.4f}, 容忍回归 eps={eps} -> {"通过" if ok else "退步过多"}'
    return ok, reason

# 候选 0.86 过了 0.85 阈值，但基线是 0.92 -> 主动退步 6 点，应拦
ok_a, ra = rule_no_regression(0.86, 0.92, eps=0.005, mode='max')
# 候选 0.918 vs 基线 0.913 -> 小幅提升，通过
ok_b, rb = rule_no_regression(0.918, 0.913, eps=0.005, mode='max')
# 候选掉 0.003，但 eps=0.005 容忍（换来了更小的模型）-> 通过
ok_c, rc = rule_no_regression(0.910, 0.913, eps=0.005, mode='max')
print('0.86 vs 0.92:', ok_a, '|', ra)
print('0.918 vs 0.913:', ok_b, '|', rb)
print('0.910 vs 0.913:', ok_c, '|', rc)
assert (not ok_a) and ok_b and ok_c
print('✅ 回归检测正确：守住「别退步」，但允许在 eps 内的权衡')

## 4 · bootstrap：给「两模型之差」装上误差棒

核心问题：候选 0.918 vs 基线 0.913，差 0.005 是真的还是噪声？

**bootstrap**：把测试集当总体，有放回重采样 B 次，每次重算 Δ=acc_cand-acc_base，得到 Δ 的分布与 95% 置信区间。
**CI 整个 > 0 → 显著更好；CI 跨 0 → 不显著**。

In [ ]:
def bootstrap_ci_diff(correct_cand, correct_base, B=2000, alpha=0.05, seed=0):
    '''correct_*: 每条测试样本是否被预测对的 0/1 数组（同一批样本！）。
       返回 (delta_点估计, ci_low, ci_high).'''
    r = np.random.default_rng(seed)
    n = len(correct_cand)
    assert len(correct_base) == n, '必须在同一批测试样本上配对比较'
    deltas = np.empty(B)
    for b in range(B):
        idx = r.integers(0, n, size=n)             # 有放回重采样下标
        deltas[b] = correct_cand[idx].mean() - correct_base[idx].mean()
    point = correct_cand.mean() - correct_base.mean()
    lo, hi = np.percentile(deltas, [100*alpha/2, 100*(1-alpha/2)])
    return point, lo, hi

# 造两个模型在同一批 2000 条测试样本上的对错记录
n = 2000
truth = rng.integers(0, 2, size=n)
# 基线 acc≈0.88，候选 acc≈0.885（真实只高一点点）
base_correct = (rng.random(n) < 0.880).astype(int)
cand_correct = (rng.random(n) < 0.885).astype(int)
point, lo, hi = bootstrap_ci_diff(cand_correct, base_correct, B=2000)
print(f'Δ 点估计 = {point:+.4f}, 95% CI = [{lo:+.4f}, {hi:+.4f}]')
significant = lo > 0
print('CI 是否整个 > 0（显著更好）?', significant)
# 真实差距只有 ~0.005，2000 条上多半不显著（CI 跨 0）—— 这正是 bootstrap 的诚实之处
assert lo <= point <= hi, '点估计应落在 CI 内'
print('✅ bootstrap CI 算出，且诚实地反映了「这点差异可能不显著」')

In [ ]:
# 对比：当候选真的大幅更好（0.95 vs 0.80），CI 应整个 > 0（显著）
base2 = (rng.random(n) < 0.80).astype(int)
cand2 = (rng.random(n) < 0.95).astype(int)
p2, lo2, hi2 = bootstrap_ci_diff(cand2, base2, B=2000)
print(f'大差距: Δ={p2:+.4f}, 95% CI=[{lo2:+.4f}, {hi2:+.4f}]')
assert lo2 > 0, '真实大幅提升时，CI 应整个为正 -> 判显著'
print('✅ 真提升 -> CI 整个>0 判显著；小到可能是噪声 -> CI 跨0 判不显著')

## 5 · 置换检验 + 把规则聚合成门禁

**置换检验**给 p 值：在「两模型无差异」零假设下，随机互换两者预测，看观测差异有多极端。

再把所有规则**聚合**：一票否决，汇总全部失败理由。

In [ ]:
def permutation_test_diff(correct_cand, correct_base, B=2000, seed=0):
    '''返回 p 值：观测 Δ 在零假设下被随机超越的频率（单侧：候选更好）。'''
    r = np.random.default_rng(seed)
    obs = correct_cand.mean() - correct_base.mean()
    pooled = np.concatenate([correct_cand, correct_base])
    n = len(correct_cand)
    count = 0
    for _ in range(B):
        perm = r.permutation(pooled)               # 打乱后重新分两组
        d = perm[:n].mean() - perm[n:].mean()
        if d >= obs:
            count += 1
    return (count + 1) / (B + 1)                   # +1 平滑，避免 p=0

p_small = permutation_test_diff(cand_correct, base_correct, B=2000)   # 真实差小
p_big = permutation_test_diff(cand2, base2, B=2000)                   # 真实差大
print(f'小差距 p 值 = {p_small:.3f}  (大 -> 不显著)')
print(f'大差距 p 值 = {p_big:.3f}  (≈0 -> 显著)')
assert p_big < 0.05, '真实大差距应显著 (p<0.05)'
assert p_small > p_big, '差距越小，p 值越大（越可能是噪声）'
print('✅ 置换检验给出 p 值，与 bootstrap CI 结论一致')

In [ ]:
# 把规则聚合成门禁：一票否决 + 汇总所有失败理由
class Decision:
    def __init__(self, passed, reasons): self.passed=passed; self.reasons=reasons

def run_gate(rules):
    '''rules: [(name, passed, reason)]; 全过才放行，汇总所有理由。'''
    failed = [(n, r) for (n, ok, r) in rules if not ok]
    reasons = [f'{n}: {r}' for (n, ok, r) in rules]
    return Decision(passed=(len(failed) == 0), reasons=reasons)

# 一个候选模型过门禁：阈值过、回归过、显著(用大差距那对)
rules = [
    ('abs_acc',) + rule_abs_threshold(0.95, 0.85, 'max'),
    ('no_reg',) + rule_no_regression(0.95, 0.80, eps=0.005, mode='max'),
    ('significant', lo2 > 0, f'95% CI=[{lo2:+.4f},{hi2:+.4f}] {"整个>0" if lo2>0 else "跨0"}'),
]
dec = run_gate(rules)
print('放行?', dec.passed)
for r in dec.reasons: print('  -', r)
assert dec.passed, '阈值+回归+显著都过，应放行'

# 反例：一个退步的候选，应被拦且理由清晰
bad = [('abs_acc',)+rule_abs_threshold(0.86,0.85,'max'),
       ('no_reg',)+rule_no_regression(0.86,0.92,eps=0.005,mode='max')]
dec_bad = run_gate(bad)
assert not dec_bad.passed, '退步候选必须被拦'
print('\n反例被正确拦截 ✅，失败理由:', [r for n,ok,r in bad if not ok])
print('✅ 门禁聚合正确：一票否决 + 汇总理由')

---
## ✏️ 练习 1：bootstrap 单指标置信区间

上面对「两模型之差」做了 bootstrap。现在对**单个**指标做：实现 `bootstrap_ci(correct, B=2000, alpha=0.05, seed=0)`，返回该指标（准确率）的点估计与 95% 置信区间 `(point, lo, hi)`。

In [ ]:
def bootstrap_ci(correct, B=2000, alpha=0.05, seed=0):
    # TODO: 有放回重采样 correct（0/1 数组）B 次，每次算 mean；
    #       point=correct.mean()，CI 取重采样 means 的 [alpha/2, 1-alpha/2] 百分位
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
correct = (rng.random(1500) < 0.90).astype(int)
pt, lo, hi = bootstrap_ci(correct, B=2000)
assert lo < pt < hi, '点估计应在 CI 内'
assert abs(pt - 0.90) < 0.03, '点估计应接近真实 0.90'
assert hi - lo < 0.06, '1500 样本上 95% CI 宽度应较窄(约±1.5%)'
assert 0.88 < pt < 0.92
print(f'acc 点估计={pt:.3f}, 95% CI=[{lo:.3f}, {hi:.3f}]')
print('✅ 练习 1 通过：单指标 bootstrap CI')

## ✏️ 练习 2：CI 宽度随样本量收缩

统计直觉：测试集越大，置信区间越窄（越确定）。实现 `ci_width(n, true_acc=0.9, B=1000, seed=0)`：
在 `n` 条样本上做 bootstrap，返回 95% CI 的宽度 `hi-lo`。然后自测会验证 n 越大宽度越小（约 1/√n）。

In [ ]:
def ci_width(n, true_acc=0.9, B=1000, seed=0):
    # TODO: 用 default_rng(seed) 造 n 条 (random<true_acc) 的 0/1 数组，
    #       对它 bootstrap，返回 95% CI 宽度 (hi-lo)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
w_small = ci_width(200, seed=1)
w_large = ci_width(5000, seed=1)
print(f'n=200  CI 宽度 = {w_small:.4f}')
print(f'n=5000 CI 宽度 = {w_large:.4f}')
assert w_large < w_small, '样本越多，CI 越窄（越确定）'
# 大致符合 1/sqrt(n)：样本 ×25，宽度约缩小 5 倍
assert w_small / w_large > 2.5, '宽度收缩应大致符合 1/sqrt(n)'
print('✅ 练习 2 通过：CI 宽度随 n 收缩（更多数据 = 更确定）')

## ✏️ 练习 3：完整门禁决策函数

把三条规则组装成一个端到端门禁。实现 `evaluate_gate(cand_acc, cand_lat, base_acc, base_correct, cand_correct)`：
规则 = ① acc ≥ 0.85；② 延迟 ≤ 100；③ 相对基线不回归(eps=0.005)；④ 提升显著(bootstrap CI 下界>0)。
返回 `(passed: bool, n_failed: int)`。

In [ ]:
def evaluate_gate(cand_acc, cand_lat, base_acc, base_correct, cand_correct):
    # TODO: 调用上面四个规则/检验，收集 (passed) 列表，
    #       passed = 全部通过；n_failed = 失败规则数。返回 (passed, n_failed)
    #       显著性用 bootstrap_ci_diff(cand_correct, base_correct) 的 lo>0
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 好候选：大幅真提升 + 低延迟 -> 全过
ok, nf = evaluate_gate(0.95, 80, 0.80, base2, cand2)
assert ok and nf == 0, f'好候选应全过, 得到 passed={ok} n_failed={nf}'
# 坏候选：退步 + 高延迟 -> 多项失败
ok2, nf2 = evaluate_gate(0.86, 150, 0.92, base2, base2)   # 候选==基线->不显著
assert (not ok2) and nf2 >= 2, f'坏候选应被拦且多项失败, 得到 {ok2},{nf2}'
print(f'好候选: passed={ok}, 失败数={nf}')
print(f'坏候选: passed={ok2}, 失败数={nf2}')
print('✅ 练习 3 通过：端到端门禁决策正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def bootstrap_ci(correct, B=2000, alpha=0.05, seed=0):
    r = np.random.default_rng(seed)
    correct = np.asarray(correct)
    n = len(correct)
    means = np.array([correct[r.integers(0, n, n)].mean() for _ in range(B)])
    point = correct.mean()
    lo, hi = np.percentile(means, [100*alpha/2, 100*(1-alpha/2)])
    return point, lo, hi

In [ ]:
# 练习 2 参考答案
def ci_width(n, true_acc=0.9, B=1000, seed=0):
    r = np.random.default_rng(seed)
    correct = (r.random(n) < true_acc).astype(int)
    _, lo, hi = bootstrap_ci(correct, B=B, seed=seed)
    return hi - lo

In [ ]:
# 练习 3 参考答案
def evaluate_gate(cand_acc, cand_lat, base_acc, base_correct, cand_correct):
    checks = []
    checks.append(rule_abs_threshold(cand_acc, 0.85, 'max')[0])
    checks.append(rule_abs_threshold(cand_lat, 100, 'min')[0])
    checks.append(rule_no_regression(cand_acc, base_acc, eps=0.005, mode='max')[0])
    _, lo, _ = bootstrap_ci_diff(cand_correct, base_correct, B=2000)
    checks.append(lo > 0)
    n_failed = sum(1 for x in checks if not x)
    return (n_failed == 0, n_failed)

---
## 🧪 真实数据胶囊：两个分类器的显著性对比

用一个**真实的二分类设定**：从两个类别（均值不同的高斯）采样特征，训练两个不同强度的阈值分类器，在同一个测试集上用我们的门禁 + bootstrap 判断「分类器 B 是否显著优于 A」。这就是 champion/challenger 的完整流程。

In [ ]:
# 真实二分类数据：两类一维高斯
def make_classification(n, sep, seed):
    r = np.random.default_rng(seed)
    y = r.integers(0, 2, size=n)
    x = np.where(y == 1, sep/2, -sep/2) + r.normal(0, 1, size=n)
    return x, y

x_tr, y_tr = make_classification(4000, sep=1.5, seed=10)
x_te, y_te = make_classification(3000, sep=1.5, seed=11)

# 两个分类器：A 用简单阈值 0；B 用训练集学到的最优阈值（两类均值中点）
thr_A = 0.0
thr_B = 0.5 * (x_tr[y_tr==1].mean() + x_tr[y_tr==0].mean())
predA = (x_te > thr_A).astype(int); correctA = (predA == y_te).astype(int)
predB = (x_te > thr_B).astype(int); correctB = (predB == y_te).astype(int)
print(f'分类器 A (thr=0)    acc = {correctA.mean():.4f}')
print(f'分类器 B (学到阈值) acc = {correctB.mean():.4f}')

point, lo, hi = bootstrap_ci_diff(correctB, correctA, B=3000)
pval = permutation_test_diff(correctB, correctA, B=3000)
print(f'B-A: Δ={point:+.4f}, 95% CI=[{lo:+.4f},{hi:+.4f}], 置换检验 p={pval:.3f}')
# 两个阈值都接近最优(数据对称)，差异通常很小、不显著 —— 门禁会诚实地说「证据不足」
assert lo <= point <= hi
decision = '显著，B 可晋升' if lo > 0 else '不显著，证据不足，维持现役 A'
print('门禁判定:', decision)
print('✅ 胶囊跑通：真实分类器对比 + 显著性判定')

**🧪 胶囊练习**：实现 `is_significantly_better(correctB, correctA, B=2000)`：综合 bootstrap（CI 下界>0）**与** 置换检验（p<0.05），**两者都满足**才返回 `True`（更保守、更可信）。

In [ ]:
def is_significantly_better(correctB, correctA, B=2000):
    # TODO: 调 bootstrap_ci_diff 得 lo；调 permutation_test_diff 得 p；
    #       返回 (lo > 0) and (p < 0.05)
    raise NotImplementedError

In [ ]:
# 自测
# 构造一个 B 明显更强的场景，确保判显著
cB = (rng.random(3000) < 0.92).astype(int)
cA = (rng.random(3000) < 0.80).astype(int)
assert is_significantly_better(cB, cA) == True, '大幅真提升应判显著'
assert is_significantly_better(cA, cA) == False, '自己 vs 自己不可能显著'
print('✅ 胶囊练习通过：bootstrap 与置换检验双重确认才放行')

In [ ]:
# 📖 胶囊参考答案
def is_significantly_better(correctB, correctA, B=2000):
    _, lo, _ = bootstrap_ci_diff(correctB, correctA, B=B)
    p = permutation_test_diff(correctB, correctA, B=B)
    return (lo > 0) and (p < 0.05)

---
## 🔧 旁注：这套东西在真实 CI/CD 里的样子

你写的门禁对应 ML CI/CD 流水线里的一个 **gate stage**（伪代码，**本环境不跑**）：

```yaml
# .github/workflows/ml-gate.yml  (概念示意)
- name: evaluate-model
  run: python eval.py --candidate model.npz --testset test@v3   # 产出 metrics
- name: gate
  run: python gate.py --metrics metrics.json --baseline champion  # == 我们的 run_gate
       # 内部：绝对阈值 + 回归 + bootstrap 显著性；不过则 exit 1，CI 变红，阻止合并/部署
- name: promote                       # 仅在 gate 通过时
  run: dvc/mlflow promote champion <new-version>   # == 模块02 的 tag 移动
```

对应关系：gate stage↔`run_gate`、bootstrap↔显著性检验、promote↔模块 02 标签。生产里还会接 canary/shadow 做线上验证。

### 小结
- 指标是随机变量：**1% 量级的差异可能纯属采样噪声**，看点估计选模型是业余。
- 门禁三件套：**绝对阈值**(守底线 SLO) + **回归检测**(别退步，eps 容忍权衡) + **显著性**(提升是真的吗)。
- **bootstrap**：重采样测试集得指标差值的 95% CI；CI 整个>0 才判显著。**置换检验**给互补的 p 值。
- 聚合 = 一票否决 + 汇总理由；过了离线门禁还要 **canary/shadow** 渐进放行（回退靠模块 02 标签）。

下一站：**模块 04 · 监控与漂移** —— 上线三个月了，它还准吗？没有标签我怎么知道？